In [ ]:
!pip install -q langchain langchain-community langchain-core
!pip install -q faiss-cpu sentence-transformers
!pip install -q gradio
!pip install -q groq
!pip install -q datasets pandas numpy scikit-learn

In [ ]:
import os

# Set your Groq API key here
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Example: create a sample document
with open("sample.txt", "w") as f:
    f.write("""
    Artificial Intelligence (AI) is the simulation of human intelligence processes by machines.
    Machine learning is a subset of AI that focuses on learning patterns from data.
    Deep learning uses neural networks with many layers.
    """)

loader = TextLoader("sample.txt")
documents = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

print(f"Total chunks: {len(docs)}")

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embedding_model)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
from langchain.llms.base import LLM
from groq import Groq

class GroqLLM(LLM):
    def __init__(self, model="llama-3.1-8b-instant"):
        self.client = Groq()
        self.model = model

    def _call(self, prompt, stop=None):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    @property
    def _llm_type(self):
        return "groq_llm"

llm = GroqLLM()

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

def rag_pipeline(query):
    retrieved_docs = retriever.get_relevant_documents(query)
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    prompt = prompt_template.format(context=context, question=query)
    answer = llm(prompt)

    return answer, retrieved_docs

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Sample evaluation dataset
eval_data = [
    {
        "question": "What is AI?",
        "ground_truth": "AI is the simulation of human intelligence by machines."
    },
    {
        "question": "What is deep learning?",
        "ground_truth": "Deep learning uses neural networks with many layers."
    }
]

In [ ]:
def evaluate_rag(eval_data):
    results = []

    for item in eval_data:
        q = item["question"]
        gt = item["ground_truth"]

        answer, retrieved_docs = rag_pipeline(q)

        # --- Retrieval Recall ---
        retrieved_text = " ".join([doc.page_content for doc in retrieved_docs])
        recall = 1 if gt.lower() in retrieved_text.lower() else 0

        # --- Embedding Similarity ---
        emb1 = embedder.encode([answer])[0]
        emb2 = embedder.encode([gt])[0]

        similarity = cosine_similarity([emb1], [emb2])[0][0]

        results.append({
            "question": q,
            "answer": answer,
            "similarity": similarity,
            "recall": recall
        })

    avg_similarity = np.mean([r["similarity"] for r in results])
    avg_recall = np.mean([r["recall"] for r in results])

    return results, avg_similarity, avg_recall

In [ ]:
results, avg_sim, avg_recall = evaluate_rag(eval_data)

print("Results:")
for r in results:
    print(r)

print("\nAverage Similarity:", avg_sim)
print("Average Recall:", avg_recall)

In [ ]:
import gradio as gr

def gradio_rag(query):
    answer, docs = rag_pipeline(query)
    sources = "\n\n".join([doc.page_content for doc in docs])

    return answer, sources

interface = gr.Interface(
    fn=gradio_rag,
    inputs=gr.Textbox(label="Ask a question"),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Retrieved Context")
    ],
    title="Naive RAG System",
    description="Ask questions based on uploaded documents"
)

interface.launch()